# Custodian DNS tunnelling model improvement

Google-hosted Colab only. Never connect a local runtime, mount Google Drive, resolve dataset domains, replay traffic, execute payloads, or trust/export a model automatically. Development, diagnostic, and fresh final-evaluation roles remain separate.


In [ ]:
import os
import re
import subprocess
import sys
from pathlib import Path

CUSTODIAN_COMMIT = ""  # Reviewed full 40-character Git SHA.
HOSTED_ACK = ""  # Required: I AM USING A HOSTED GOOGLE COLAB RUNTIME

if HOSTED_ACK != "I AM USING A HOSTED GOOGLE COLAB RUNTIME":
    raise RuntimeError("Hosted-runtime acknowledgement is required.")
if not any(os.environ.get(name) for name in ("COLAB_RELEASE_TAG", "COLAB_BACKEND_VERSION")):
    raise RuntimeError("Hosted Colab markers are absent; local runtime is forbidden.")
if os.environ.get("CUSTODIAN_ALLOW_LOCAL_RUNTIME"):
    raise RuntimeError("Local-runtime overrides are forbidden.")
if re.fullmatch(r"[0-9a-f]{40}", CUSTODIAN_COMMIT.strip().lower()) is None:
    raise ValueError("Use a complete reviewed commit SHA.")
WORKSPACE = Path("/content/custodian-workspace")
REPO = WORKSPACE / "repository"
if WORKSPACE.exists():
    raise FileExistsError("Refusing to reuse an existing workspace.")
subprocess.run(
    ["git", "clone", "--no-checkout", "https://github.com/EmberFalls/Custodian.git", str(REPO)],
    check=True,
)
subprocess.run(["git", "checkout", "--detach", CUSTODIAN_COMMIT], cwd=REPO, check=True)
assert (
    subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO, text=True).strip()
    == CUSTODIAN_COMMIT
)

In [ ]:
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-r", str(REPO / "training/requirements-colab.txt")],
    check=True,
)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "--no-deps", "--editable", str(REPO)], check=True
)
sys.path[:0] = [str(REPO), str(REPO / "src")]
os.chdir(REPO)
subprocess.run([sys.executable, "-m", "pytest", "-q"], check=True)

## Prepare development and diagnostic roles

The fifteen development files alone feed grouped train, validation, calibration, and internal-test partitions. The three diagnostic files are separate and may not be used to retune the completed candidate. Preparation uses the same `dns.v1` extractor as runtime.


In [ ]:
from training.run_dns_colab import prepare, prepare_external_holdout

PREPARED, PROVENANCE = prepare(WORKSPACE)
DIAGNOSTIC, DIAGNOSTIC_PROVENANCE = prepare_external_holdout(WORKSPACE)
print("Development and diagnostic preparation complete; no model has been fit.")

## Lock the fresh external source before fitting

The immutable DNS Threats Dataset v1 test split is downloaded from Zenodo, verified by its published MD5 and Custodian's locked SHA-256, and adapted from raw `domain,class` rows. Class 1 (DGA) is excluded, not relabelled. The prepared final table is not passed to candidate selection. DNSTunnel2026 was separately rejected because its release contains processed source-specific features rather than its described raw schema.


In [ ]:
from training.ctu_dns import prepare_ctu_final_evaluation

FINAL_DATA, FINAL_PROVENANCE = prepare_ctu_final_evaluation(WORKSPACE)
print("Fresh external source locked before model fit; candidate code has not accessed its rows.")

## Fit candidates and run the non-tuning diagnostic

Candidate choice, calibration, and the DNS-tunnel threshold use development partitions only. The diagnostic result is recorded after selection and cannot alter this package.


In [ ]:
from training.run_dns_colab import evaluate_external_holdout, fit

FIT_CONFIRMATION = ""  # Required: FIT DNS CANDIDATES
if FIT_CONFIRMATION != "FIT DNS CANDIDATES":
    raise RuntimeError("Explicit fit confirmation is required.")

PACKAGE = fit(PREPARED, PROVENANCE, WORKSPACE / "output/dns_candidate")
DIAGNOSTIC_METRICS = evaluate_external_holdout(PACKAGE, DIAGNOSTIC, DIAGNOSTIC_PROVENANCE)
print("Candidate remains untrusted; diagnostic data was not used for retuning.")

## One-time fresh external evaluation

Run only after the candidate package is final. The evaluator refuses a second run for the same package and never fits, recalibrates, selects, or changes a threshold. Passing targets still requires human review and does not prove live-network performance.


In [ ]:
from training.run_dns_colab import evaluate_fresh_external_once

FINAL_EVALUATION_CONFIRMATION = ""  # Required: RUN FRESH EXTERNAL EVALUATION ONCE
if FINAL_EVALUATION_CONFIRMATION != "RUN FRESH EXTERNAL EVALUATION ONCE":
    raise RuntimeError("Explicit one-time final-evaluation confirmation is required.")

FINAL_METRICS = evaluate_fresh_external_once(PACKAGE, FINAL_DATA, FINAL_PROVENANCE)
print({"targets_met": FINAL_METRICS["predeclared_external_targets_met"], "trusted": False})
print("Review all reports and hashes. Do not enable configs/models.yaml automatically.")